# Jeux globaux — la sélection d'équilibre par le bruit privé (Version C#)

*Carnet 11f de la série GameTheory — portage C# (BCL) du jumeau Python ; prolonge 11 (jeux bayésiens) et 10 (induction en avant). Les valeurs déterministes sont recalculées et assertées ; les tirages Monte-Carlo et les figures du jumeau Python sont cités, non rejoués.*

**Navigation** : [<< 11-BayesianGames](GameTheory-11-BayesianGames-Python.ipynb) | [Version Python](GameTheory-11f-Jeux-Globaux-Selection-Equilibre-Python.ipynb) | [Index](README.md)

Un jeu de **coordination** possède, à information complète, plusieurs équilibres : rien dans les payoffs ne désigne celui qui sera joué. Les **jeux globaux** (Carlsson et van Damme, 1993) lèvent cette indétermination en perturbant l'information : l'état de l'économie n'est pas observé exactement, mais à travers un **signal privé bruité**. Deux résultats en découlent, et ce carnet les mesure :

1. l'équilibre devient **unique**, et prend la forme d'une **stratégie de coupure** ;
2. quand le bruit s'éteint, cette coupure converge vers l'équilibre **risque-dominant** au sens de Harsanyi et Selten — et non vers l'équilibre Pareto-dominant, avec lequel il entre en conflit sur une moitié de la bande de multiplicité.

C'est une sélection obtenue par la **croyance**, non par le temps. L'induction en avant du carnet 10 opère l'autre sélection, le long de l'arbre : là, chaque nœud est visité et la crédibilité tranche ; ici, tout est simultané et seule la finesse du signal tranche.

**Plan.** §1 le jeu et sa multiplicité · §2 la perturbation · §3 la coupure unique · §4 ce que la limite sélectionne, mesuré · §5 la sélection sous coordination perturbée, mesurée · §6 le raccourci qui ne marche pas · §7 exercices.


## 1. Le jeu — une complémentarité stratégique

Deux joueurs (`i = 1, 2`) choisissent **simultanément** d'**investir** (`I`) ou de s'**abstenir** (`N`). L'état `θ` est le rendement fondamental de l'investissement. Investir coûte `c`, rapporte `θ`, et procure en plus un bonus `γ` **si l'autre investit aussi** :

$$
u_i(I, I) = \theta - c + \gamma, \qquad u_i(I, N) = \theta - c, \qquad u_i(N, \cdot) = 0
$$

Le terme `γ` est la **complémentarité stratégique** : investir vaut davantage quand l'autre investit. C'est lui, et lui seul, qui crée la multiplicité — sans lui, chacun décide seul de son côté, l'investissement est dominé ou dominant selon `θ`, et il n'y a rien à sélectionner.

À information complète (`θ` connu de tous), on vérifie chaque équilibre par **déviation** : `(I, I)` tient si dévier vers `N` ne paie pas, c'est-à-dire si `u(I, I) >= u(N, I) = 0` ; `(N, N)` tient si `u(N, N) = 0 >= u(I, N)`.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

public static class Gt11f
{
    public const double MU = 0.8;
    public const double SIGMA_T = 0.5;
    public const double C = 1.0;
    public const double GAMMA = 0.6;

    // Assertion executable : leve si la valeur recalculee s'ecarte de l'attendu.
    public static void AssertClose(double actual, double expected, double tol, string label)
    {
        double e = Math.Abs(actual - expected);
        if (e > tol)
            throw new Exception($"ASSERTION {label} : attendu {expected}, obtenu {actual}, ecart {e}");
    }

    // erf : serie de Taylor (precise a ~1e-15 pres pour |x| < 2) ;
    // A&S 7.1.26 (erreur 1.5e-7) au dela, ou l'ecart d'indifference est loin de sa racine.
    public static double Erf(double x)
    {
        double s = x < 0 ? -1.0 : 1.0;
        double ax = Math.Abs(x);
        if (ax >= 2.0)
        {
            double t = 1.0 / (1.0 + 0.3275911 * ax);
            double y = 1.0 - (((((1.061405429 * t - 1.453152027) * t) + 1.421413741) * t - 0.284496736) * t + 0.254829592) * t * Math.Exp(-ax * ax);
            return s * y;
        }
        double term = ax;
        double sum = term;
        for (int n = 1; n < 40; n++)
        {
            term *= -ax * ax * (2.0 * n - 1.0) / (n * (2.0 * n + 1.0));
            double ns = sum + term;
            if (ns == sum) break;
            sum = ns;
        }
        return s * sum * (2.0 / Math.Sqrt(Math.PI));
    }

    public static double Phi(double z) => 0.5 * (1.0 + Erf(z / Math.Sqrt(2.0)));

    public static double Payoff(string action, string autre, double theta, double c = C, double gamma = GAMMA)
    {
        if (action == "I")
            return theta - c + gamma * (autre == "I" ? 1.0 : 0.0);
        return 0.0;
    }

    public static (double Lam, double VCond) Coefficients(double se)
    {
        double st2 = SIGMA_T * SIGMA_T;
        double se2 = se * se;
        double lam = st2 / (st2 + se2);
        double vcond = st2 * se2 / (st2 + se2);
        return (lam, vcond);
    }

    public static double EsperanceEtat(double x, double se)
    {
        double lam = Coefficients(se).Lam;
        return MU + lam * (x - MU);
    }

    public static double ProbaAutre(double cstar, double se)
    {
        var cf = Coefficients(se);
        double muTheta = MU + cf.Lam * (cstar - MU);
        double w = (muTheta - cstar) / Math.Sqrt(cf.VCond + se * se);
        return Phi(w);
    }

    public static double EcartIndifference(double x, double se)
    {
        return EsperanceEtat(x, se) - C + GAMMA * ProbaAutre(x, se);
    }

    public static double Coupure(double se, double lo = -20.0, double hi = 20.0, double tol = 1e-13)
    {
        if (!(EcartIndifference(lo, se) < 0.0 && EcartIndifference(hi, se) > 0.0))
            throw new ArgumentException("pas d'encadrement de la racine");
        for (int i = 0; i < 300; i++)
        {
            double milieu = 0.5 * (lo + hi);
            double v = EcartIndifference(milieu, se);
            if (Math.Abs(v) < tol || (hi - lo) < tol) return milieu;
            if (v < 0.0) lo = milieu; else hi = milieu;
        }
        return 0.5 * (lo + hi);
    }

    public static double MeilleureReponse(double xAutre, double se, double lo = -20.0, double hi = 20.0, double tol = 1e-13)
    {
        var cf = Coefficients(se);
        double Ecart(double x)
        {
            double muTheta = MU + cf.Lam * (x - MU);
            double w = (muTheta - xAutre) / Math.Sqrt(cf.VCond + se * se);
            return muTheta - C + GAMMA * Phi(w);
        }
        if (!(Ecart(lo) < 0.0 && Ecart(hi) > 0.0))
            return double.NaN;
        for (int i = 0; i < 300; i++)
        {
            double milieu = 0.5 * (lo + hi);
            double v = Ecart(milieu);
            if (Math.Abs(v) < tol || (hi - lo) < tol) return milieu;
            if (v < 0.0) lo = milieu; else hi = milieu;
        }
        return 0.5 * (lo + hi);
    }

    public static (double Pii, double Pnn) ProduitsDeNash(double theta)
    {
        double uIi = theta - C + GAMMA;
        double uIn = theta - C;
        double perteIi = uIi - 0.0;
        double perteNn = 0.0 - uIn;
        return (perteIi * perteIi, perteNn * perteNn);
    }

    public static double TauxInduit(double pi, double theta, double se)
    {
        double lam = Coefficients(se).Lam;
        double xStar = MU + (C - GAMMA * pi - MU) / lam;
        return Phi((theta - xStar) / se);
    }

    public static List<double> Trajectoire(double theta, double se, double pi0, int maxIter = 500, double tol = 1e-13)
    {
        var suite = new List<double> { pi0 };
        for (int i = 0; i < maxIter; i++)
        {
            suite.Add(TauxInduit(suite[suite.Count - 1], theta, se));
            if (Math.Abs(suite[suite.Count - 1] - suite[suite.Count - 2]) < tol)
                break;
        }
        return suite;
    }
}


var inv = CultureInfo.InvariantCulture;
Console.WriteLine($"mu = {Gt11f.MU.ToString(inv)}   sigma_theta = {Gt11f.SIGMA_T.ToString(inv)}   c = {Gt11f.C.ToString(inv)}   gamma = {Gt11f.GAMMA.ToString(inv)}");
Console.WriteLine($"bande de multiplicite attendue (c - gamma, c) = ({(Gt11f.C - Gt11f.GAMMA).ToString("F2", inv)}, {Gt11f.C.ToString("F2", inv)})");
Console.WriteLine();
Console.WriteLine("  theta   u(I,I)   u(I,N)  (I,I) EN  (N,N) EN  regime");
foreach (double theta in new[] { 0.2, 0.4, 0.6, 0.8, 1.0, 1.3 })
{
    double uIi = Gt11f.Payoff("I", "I", theta);
    double uIn = Gt11f.Payoff("I", "N", theta);
    bool eqIi = uIi >= 0.0;
    bool eqNn = 0.0 >= uIn;
    string regime = (eqIi && eqNn) ? "deux equilibres" : (eqIi ? "(I,I) unique" : "(N,N) unique");
    Console.WriteLine($"{theta.ToString("F2", inv),7} {uIi.ToString("F2", inv),8} {uIn.ToString("F2", inv),8} {eqIi,9} {eqNn,9}  {regime}");
}
Gt11f.AssertClose(Gt11f.Payoff("I", "I", 0.2), -0.20, 1e-9, "u(I,I)@0.2");
Gt11f.AssertClose(Gt11f.Payoff("I", "N", 0.4), -0.60, 1e-9, "u(I,N)@0.4");
Gt11f.AssertClose(Gt11f.Payoff("I", "I", 1.3), 0.90, 1e-9, "u(I,I)@1.3");
Gt11f.AssertClose(Gt11f.Payoff("I", "N", 1.3), 0.30, 1e-9, "u(I,N)@1.3");
Console.WriteLine("OK - assertions cellule 2");


The below script needs to be able to find the current output cell; this is an easy method to get it.

mu = 0.8   sigma_theta = 0.5   c = 1   gamma = 0.6


bande de multiplicite attendue (c - gamma, c) = (0.40, 1.00)


  theta   u(I,I)   u(I,N)  (I,I) EN  (N,N) EN  regime


   0.20    -0.20    -0.80     False      True  (N,N) unique


   0.40     0.00    -0.60      True      True  deux equilibres


   0.60     0.20    -0.40      True      True  deux equilibres


   0.80     0.40    -0.20      True      True  deux equilibres


   1.00     0.60     0.00      True      True  deux equilibres


   1.30     0.90     0.30      True     False  (I,I) unique


OK - assertions cellule 2


### Lecture du résultat

Trois régimes, séparés par deux seuils :

| `θ` | équilibres | pourquoi |
|---|---|---|
| `θ < c - γ` | `(N, N)` seul | investir perd même si l'autre investit (`u(I, I) < 0`) : `I` est **dominée** |
| `c - γ ≤ θ ≤ c` | **`(I, I)` et `(N, N)`** | chacun veut investir si l'autre investit, et s'abstenir sinon : **deux** équilibres |
| `θ > c` | `(I, I)` seul | investir gagne même si l'autre s'abstient (`u(I, N) > 0`) : `I` est **dominante** |

La bande du milieu est le problème. Sa largeur vaut exactement `γ = 0.60`, et à l'intérieur la théorie est muette : rien, dans les payoffs, ne hiérarchise `(I, I)` et `(N, N)`. Aux deux bornes un équilibre devient faible — `(I, I)` en `θ = c - γ`, `(N, N)` en `θ = c` — ce qui est la signature d'un changement de régime et non d'une indétermination.

Toute la question des jeux globaux tient en une phrase : **que devient cette bande quand les joueurs cessent de connaître `θ` exactement ?**

## 2. La perturbation — un état commun, des signaux privés

`θ` n'est plus connu : c'est un tirage, et **chaque joueur en reçoit un signal bruité** :

$$
\theta \sim \mathcal{N}(\mu, \sigma_\theta^2), \qquad x_i = \theta + \sigma_\varepsilon \, \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, 1) \ \text{i.i.d.}
$$

Le bruit est **privé** : il est indépendant d'un joueur à l'autre. Chacun sait que l'autre détient *un* signal, mais pas lequel. C'est précisément cet angle mort qui fait le travail de sélection.

Vu son signal, le joueur met à jour sa croyance sur `θ` — une moyenne pondérée de l'a priori et du signal :

$$
\lambda = \frac{\sigma_\theta^2}{\sigma_\theta^2 + \sigma_\varepsilon^2}, \qquad
\mathbb{E}[\theta \mid x_i] = \mu + \lambda (x_i - \mu), \qquad
\mathrm{Var}(\theta \mid x_i) = \frac{\sigma_\theta^2 \sigma_\varepsilon^2}{\sigma_\theta^2 + \sigma_\varepsilon^2}
$$

`λ` est la **précision relative** du signal : `λ → 1` quand le bruit s'éteint, `λ → 0` quand il noie l'information.

In [2]:

var inv = CultureInfo.InvariantCulture;
Console.WriteLine(" sigma_eps    lambda  Var(theta|x)  sqrt(Var)");
foreach (double se in new[] { 1.0, 0.5, 0.3, 0.1, 0.02, 0.001 })
{
    var cf = Gt11f.Coefficients(se);
    Console.WriteLine($"{se.ToString("F3", inv),10} {cf.Lam.ToString("F6", inv),9} {cf.VCond.ToString("F6", inv),14} {Math.Sqrt(cf.VCond).ToString("F6", inv),11}");
}
Gt11f.AssertClose(Gt11f.Coefficients(1.0).Lam, 0.200000, 1e-9, "lambda@1.0");
Gt11f.AssertClose(Gt11f.Coefficients(0.5).Lam, 0.500000, 1e-9, "lambda@0.5");
Gt11f.AssertClose(Gt11f.Coefficients(0.3).Lam, 0.735294, 1e-6, "lambda@0.3");
Gt11f.AssertClose(Gt11f.Coefficients(0.1).Lam, 0.961538, 1e-6, "lambda@0.1");
Gt11f.AssertClose(Gt11f.Coefficients(1.0).VCond, 0.200000, 1e-9, "Var@1.0");
Gt11f.AssertClose(Gt11f.Coefficients(0.3).VCond, 0.066176, 1e-6, "Var@0.3");

// controle Monte-Carlo : les tirages du jumeau Python sont CITES (non rejoues ici)
Console.WriteLine();
Console.WriteLine("controle a x = 1.0, sigma_eps = 0.3 :");
double mcCite = 0.94337;
int nCite = 10494;
double formule = Gt11f.EsperanceEtat(1.0, 0.3);
Console.WriteLine($"  Monte-Carlo {mcCite.ToString("F5", inv)}  ({nCite} tirages)  [CITE du jumeau Python]");
Console.WriteLine($"  formule     {formule.ToString("F5", inv)}  [recalcule]");
Gt11f.AssertClose(formule, 0.94706, 1e-5, "E[theta|x=1.0]@se=0.3");
Console.WriteLine("OK - assertions cellule 5");


 sigma_eps    lambda  Var(theta|x)  sqrt(Var)


     1.000  0.200000       0.200000    0.447214


     0.500  0.500000       0.125000    0.353553


     0.300  0.735294       0.066176    0.257248


     0.100  0.961538       0.009615    0.098058


     0.020  0.998403       0.000399    0.019984


     0.001  0.999996       0.000001    0.001000


controle a x = 1.0, sigma_eps = 0.3 :


  Monte-Carlo 0.94337  (10494 tirages)  [CITE du jumeau Python]


  formule     0.94706  [recalcule]


OK - assertions cellule 5


### Lecture du résultat

Deux progressions se lisent ensemble. Quand le bruit passe de `1.0` à `0.001`, le poids `λ` du signal monte de `0.200` à `0.999996` et la variance conditionnelle s'effondre de `0.200` à `1e-06` : le joueur cesse d'être incertain. À l'autre bout, `λ = 0.2` dit qu'un signal deux fois plus dispersé que l'état lui-même n'apporte qu'un cinquième de l'information — le cas du joueur qui « ne sait presque rien ».

Le contrôle Monte-Carlo valide la formule de mise à jour. En simulant 400 000 couples `(θ, x)` et en ne gardant que ceux dont le signal tombe à `± 0.02` de `x = 1.0`, la moyenne empirique de `θ` vaut `0.94337` (CITÉ du jumeau Python, non rejoué ici), contre `0.94706` pour `E[θ | x] = μ + λ(x - μ)` (recalculé et asserté ici). L'écart de `0.0037` est de l'ordre de l'erreur d'échantillonnage de cette moyenne : `0.257 / √10494 ≈ 0.0025`, pour un écart-type conditionnel de `0.257` et 10 494 tirages retenus. La formule est donc la bonne, et c'est elle qui porte toute l'équation d'indifférence du paragraphe suivant.


## 3. L'équilibre à coupure unique

On cherche un équilibre **symétrique en stratégie de coupure** : chaque joueur investit si et seulement si son signal dépasse un seuil `c*`. Un tel équilibre tient dans une seule équation — l'indifférence **au seuil**.

Au signal `c*`, investir et s'abstenir doivent rapporter la même chose :

$$
\underbrace{\mathbb{E}[\theta \mid c^*]}_{\text{gain si l'autre investit}} - c + \gamma \underbrace{\mathbb{P}(x_j \geq c^* \mid x_i = c^*)}_{p} = 0
$$

Reste à calculer `p`. Sachant `x_i`, le signal de l'autre est normal, centré sur la **même** espérance `E[θ | x_i]`, avec la variance `Var(θ | x_i) + σ_ε²` — l'incertitude sur l'état **plus** le bruit propre de l'autre :

$$
p = \Phi\!\left( \frac{\mathbb{E}[\theta \mid c^*] - c^*}{\sqrt{\mathrm{Var}(\theta \mid c^*) + \sigma_\varepsilon^2}} \right) = \Phi(w),
\qquad
w = -\frac{(1 - \lambda)(c^* - \mu)}{\sqrt{\mathrm{Var}(\theta \mid c^*) + \sigma_\varepsilon^2}}
$$

La même expression sert pour **deux** joueurs comme pour un **continuum** : l'opposant unique investit avec probabilité `p`, et dans une population la fraction qui investit vaut cette même probabilité — la moyenne d'indicateurs i.i.d. est leur probabilité. La suite n'a donc pas à choisir entre les deux lectures.

Deux remarques qui portent tout le résultat :

- quand `σ_ε → 0`, on a `w → 0`, donc `p → Φ(0) = 1/2` : au seuil, un joueur croit que l'autre investit **une fois sur deux** ;
- l'équation se réduit alors à `c* - c + γ/2 = 0`, soit `c* = c - γ/2` — le **milieu** de la bande de multiplicité, et non l'un de ses bords.

In [3]:

var inv = CultureInfo.InvariantCulture;
double seuilRisque = Gt11f.C - Gt11f.GAMMA / 2.0;
Console.WriteLine($"seuil de dominance du risque   c - gamma/2 = {seuilRisque.ToString("F4", inv)}");
Console.WriteLine($"bande de multiplicite          (c - gamma, c) = ({(Gt11f.C - Gt11f.GAMMA).ToString("F2", inv)}, {Gt11f.C.ToString("F2", inv)})");
Console.WriteLine();
Console.WriteLine(" sigma_eps           c*  c* - (c-gamma/2)            p");
foreach (double se in new[] { 1.0, 0.5, 0.3, 0.2, 0.1, 0.05, 0.02, 0.01, 0.001 })
{
    double cs = Gt11f.Coupure(se);
    Console.WriteLine($"{se.ToString("F3", inv),10} {cs.ToString("F7", inv),12} {(cs - seuilRisque).ToString("E3", inv),17} {Gt11f.ProbaAutre(cs, se).ToString("F6", inv),10}");
}
Gt11f.AssertClose(seuilRisque, 0.7, 1e-9, "seuil_risque");
Gt11f.AssertClose(Gt11f.Coupure(0.001), 0.6999319, 1e-5, "c*@0.001");
Gt11f.AssertClose(Gt11f.Coupure(0.05), 0.6954522, 1e-5, "c*@0.05");
Gt11f.AssertClose(Gt11f.Coupure(0.02), 0.6984646, 1e-5, "c*@0.02");
Gt11f.AssertClose(Gt11f.Coupure(1.0), -0.8632668, 1e-4, "c*@1.0");
Console.WriteLine("OK - assertions cellule 8");


seuil de dominance du risque   c - gamma/2 = 0.7000


bande de multiplicite          (c - gamma, c) = (0.40, 1.00)


 sigma_eps           c*  c* - (c-gamma/2)            p


     1.000   -0.8632668       -1.563E+000   0.887756


     0.500    0.4740985       -2.259E-001   0.604918


     0.300    0.6261851       -7.381E-002   0.546342


... (3 outputs left out)

     0.020    0.6984646       -1.535E-003   0.502289


     0.010    0.6992780       -7.220E-004   0.501136


     0.001    0.6999319       -6.815E-005   0.500113


OK - assertions cellule 8


### Lecture du résultat

La dernière colonne est la clé. À mesure que le bruit s'éteint, la probabilité que le joueur **au seuil** attribue à l'investissement de l'autre tend vers `0.5` — elle vaut `0.5001` à `σ_ε = 0.001` — et la coupure vient buter sur `c - γ/2 = 0.7000` : à `σ_ε = 0.001` l'écart tombe à `6.8e-05`.

Deux choses sont acquises à ce stade :

- pour **chaque** niveau de bruit l'équation d'indifférence a une racine (la section suivante montre qu'elle est unique) : l'équilibre existe, et il est unique là où le jeu à information complète en admettait deux ;
- la limite `c - γ/2` n'est pas un bord de la bande mais son **milieu** : la sélection a donc un contenu précis, que le §4 identifie.

Le sens économique du résultat se lit sur la première ligne : plus le bruit est fort (`σ_ε = 1.0`), plus la coupure descend (`c* = -0.86`). Un joueur très mal informé exige un fondamental bien plus favorable avant d'investir — parce qu'il sait que l'autre, aussi mal informé, risque de s'abstenir.

In [4]:

var inv = CultureInfo.InvariantCulture;
// figure du jumeau Python retraced en table : une seule traversée de zéro par courbe
Console.WriteLine("retrace de la figure : traversée de zéro de l'ecart d'indifference");
Console.WriteLine("  se      x=-0.5      x=c*        x=+1.5     racine c*");
foreach (double se in new[] { 0.8, 0.3, 0.1, 0.02 })
{
    double cs = Gt11f.Coupure(se);
    double a = Gt11f.EcartIndifference(-0.5, se);
    double b = Gt11f.EcartIndifference(cs, se);
    double c = Gt11f.EcartIndifference(1.5, se);
    Console.WriteLine($"{se.ToString("F1", inv),5}  {a.ToString("F6", inv),9}   {b.ToString("E2", inv),9}   {c.ToString("F6", inv),9}   {cs.ToString("F6", inv),9}");
}
Gt11f.AssertClose(Gt11f.Coupure(0.02), 0.6984646, 1e-5, "c*@0.02 (retrace)");
Console.WriteLine("OK - retrace cellule 10 (figure -> table)");


retrace de la figure : traversée de zéro de l'ecart d'indifference


  se      x=-0.5      x=c*        x=+1.5     racine c*


  0.8  -0.055721   -5.41E-014    0.170101   -0.135308


  0.3  -0.671048   3.14E-014    0.506454    0.626185


  0.1  -1.066327   -9.32E-014    0.727345    0.688370


  0.0  -1.180358   -4.41E-014    0.789417    0.698465


OK - retrace cellule 10 (figure -> table)


### Lecture du résultat

La table ci-dessus retrace la figure du jumeau Python : chaque courbe **traverse zéro une seule fois** (le changement de signe de l'écart d'indifférence, nul au seuil `c*`), et le point de traversée se déplace vers `c - γ/2` à mesure que le bruit diminue. Une seule traversée signifie une seule stratégie de coupure compatible avec l'indifférence, donc **un seul** équilibre — au même endroit de la bande où le jeu à information complète en admettait deux.

La coupure tombe à l'intérieur de la bande de multiplicité : la sélection ne consiste pas à retenir un bord, mais un point **intérieur**. C'est ce point que la section suivante nomme.


In [5]:

var inv = CultureInfo.InvariantCulture;
Console.WriteLine("Croisement unique : la condition d'indifference est-elle croissante ?");
double[] grilleAudit = new double[161];
for (int i = 0; i < 161; i++) grilleAudit[i] = -8.0 + i * (16.0 / 160.0);
double[] ses = { 0.8, 0.3, 0.1, 0.02 };
foreach (double se in ses)
{
    bool croissante = true;
    double prec = Gt11f.EcartIndifference(grilleAudit[0], se);
    for (int i = 1; i < grilleAudit.Length; i++)
    {
        double cur = Gt11f.EcartIndifference(grilleAudit[i], se);
        if (cur <= prec) { croissante = false; break; }
        prec = cur;
    }
    Console.WriteLine($"  sigma_eps = {se.ToString("F3", inv)} : strictement croissante sur [-8, 8] = {croissante}");
}
Console.WriteLine();
Console.WriteLine("Unicite par iteration de la meilleure reponse, depuis quatre departs :");
var expected = new Dictionary<double, double>
{
    [0.8] = -0.1353085, [0.3] = 0.6261851, [0.1] = 0.6883698, [0.02] = 0.6984646
};
foreach (double se in ses)
{
    double[] departs = { -3.0, 0.0, 1.0, 2.5 };
    double[] arrives = new double[4];
    for (int d = 0; d < 4; d++)
    {
        double x = departs[d];
        for (int i = 0; i < 500; i++)
        {
            double suivant = Gt11f.MeilleureReponse(x, se);
            if (double.IsNaN(suivant)) { x = double.NaN; break; }
            if (Math.Abs(suivant - x) < 1e-14) { x = suivant; break; }
            x = suivant;
        }
        arrives[d] = x;
    }
    double etendue = arrives.Max() - arrives.Min();
    string arrStr = string.Join(" ", arrives.Select(v => v.ToString("F7", inv).PadLeft(12)));
    Console.WriteLine($"{se.ToString("F3", inv),10} {arrStr} {etendue.ToString("E2", inv),10}");
    Gt11f.AssertClose(arrives[0], expected[se], 1e-5, $"point fixe @se={se}");
}
Console.WriteLine("OK - assertions cellule 12");


Croisement unique : la condition d'indifference est-elle croissante ?


  sigma_eps = 0.800 : strictement croissante sur [-8, 8] = True


  sigma_eps = 0.300 : strictement croissante sur [-8, 8] = True


  sigma_eps = 0.100 : strictement croissante sur [-8, 8] = True


  sigma_eps = 0.020 : strictement croissante sur [-8, 8] = True


Unicite par iteration de la meilleure reponse, depuis quatre departs :


     0.800   -0.1353085   -0.1353085   -0.1353085   -0.1353085  5.68E-013


     0.300    0.6261851    0.6261851    0.6261851    0.6261851  1.42E-013


     0.100    0.6883698    0.6883698    0.6883698    0.6883698  1.42E-013


     0.020    0.6984646    0.6984646    0.6984646    0.6984646  4.26E-013


OK - assertions cellule 12


### Lecture du résultat

Deux mesures d'unicité, indépendantes l'une de l'autre :

- la condition d'indifférence est **strictement croissante** sur `[-8, 8]` à chaque niveau de bruit : elle ne peut donc s'annuler qu'une fois ;
- l'itération de la meilleure réponse, lancée depuis quatre points de départ éloignés (`-3`, `0`, `1`, `2.5`), aboutit au **même** point fixe, l'écart entre les trajectoires tombant à `1e-13`. Un équilibre multiple aurait offert plusieurs points fixes, chacun attirant un bassin.

C'est une mesure, pas la preuve de Carlsson et van Damme : leur résultat passe par l'**élimination itérée des stratégies dominées**, et vaut pour toute la classe des jeux globaux. Ce que ce carnet établit directement, c'est que sur ce jeu-ci il n'existe qu'une coupure.

## 4. Ce que la limite sélectionne — la dominance du risque

La limite `c - γ/2` n'est pas une constante trouvée au hasard numérique : c'est le seuil qui sépare les deux équilibres selon le critère de **Harsanyi et Selten**. Pour un jeu 2×2 à deux équilibres stricts, on compare les **produits de Nash** : pour chaque équilibre, le produit des pertes qu'encourrait chaque joueur en déviant vers sa meilleure réponse. L'équilibre au produit le plus élevé est **risque-dominant** — c'est celui qu'on choisirait si l'on croyait l'autre également susceptible de jouer l'une ou l'autre action.

Sur ce jeu, les pertes à la déviation valent :

| équilibre | payoff | déviation | perte | produit |
|---|---|---|---|---|
| `(I, I)` | `θ - c + γ` | `N` rapporte `0` | `θ - c + γ` | `(θ - c + γ)²` |
| `(N, N)` | `0` | `I` rapporte `θ - c` | `c - θ` | `(c - θ)²` |

`(I, I)` est donc risque-dominant dès que `θ - c + γ ≥ c - θ`, c'est-à-dire **`θ ≥ c - γ/2`**. Exactement la limite mesurée au §3. Le jeu global n'a donc pas seulement rendu l'équilibre unique : il a **choisi lequel** des deux le serait, et ce choix est le critère de dominance du risque.

C'est ici que les deux critères entrent en conflit. L'équilibre **Pareto-dominant** est `(I, I)` dès que `θ > c - γ` : sur la moitié basse de la bande, `c - γ ≤ θ < c - γ/2`, `(I, I)` est le meilleur pour les deux joueurs — et c'est pourtant `(N, N)` que la dominance du risque retient, et donc que le jeu global sélectionne.

In [6]:

var inv = CultureInfo.InvariantCulture;
Console.WriteLine("  theta  produit (I,I)  produit (N,N)  risque-dominant");
foreach (double theta in new[] { 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00 })
{
    var pn = Gt11f.ProduitsDeNash(theta);
    string gagnant = Math.Abs(pn.Pii - pn.Pnn) < 1e-12 ? "egalite" : (pn.Pii > pn.Pnn ? "(I,I)" : "(N,N)");
    Console.WriteLine($"{theta.ToString("F2", inv),7} {pn.Pii.ToString("F6", inv),15} {pn.Pnn.ToString("F6", inv),15}  {gagnant}");
}
double lo = Gt11f.C - Gt11f.GAMMA, hi = Gt11f.C;
for (int i = 0; i < 200; i++)
{
    double milieu = 0.5 * (lo + hi);
    var pn = Gt11f.ProduitsDeNash(milieu);
    if (pn.Pii > pn.Pnn) hi = milieu; else lo = milieu;
}
double seuilNash = 0.5 * (lo + hi);
double csFin = Gt11f.Coupure(0.001);
Console.WriteLine();
Console.WriteLine($"seuil de dominance du risque (produits de Nash) : {seuilNash.ToString("F10", inv)}");
Console.WriteLine($"c - gamma/2                                     : {seuilRisque.ToString("F10", inv)}");
Console.WriteLine($"ecart                                           : {(seuilNash - seuilRisque).ToString("E3", inv)}");
Console.WriteLine($"c* mesure a sigma_eps = 0.001                   : {csFin.ToString("F7", inv)}");
Console.WriteLine($"ecart de la mesure a c - gamma/2                : {(csFin - seuilRisque).ToString("E3", inv)}");
Console.WriteLine($"seuil de dominance Pareto (bord bas de bande)   : {(Gt11f.C - Gt11f.GAMMA).ToString("F4", inv)}");
Gt11f.AssertClose(seuilNash, 0.7000000000, 1e-9, "seuil_nash");
Gt11f.AssertClose(seuilNash - seuilRisque, 0.0, 1e-9, "ecart nash/risque");
Gt11f.AssertClose(Gt11f.C - Gt11f.GAMMA, 0.4000, 1e-9, "seuil pareto");
Console.WriteLine("OK - assertions cellule 15");


  theta  produit (I,I)  produit (N,N)  risque-dominant


   0.40        0.000000        0.360000  (N,N)


   0.50        0.010000        0.250000  (N,N)


   0.60        0.040000        0.160000  (N,N)


   0.70        0.090000        0.090000  egalite


   0.80        0.160000        0.040000  (I,I)


... (5 outputs left out)

c* mesure a sigma_eps = 0.001                   : 0.6999319


ecart de la mesure a c - gamma/2                : -6.815E-005


seuil de dominance Pareto (bord bas de bande)   : 0.4000


OK - assertions cellule 15


### Lecture du résultat

Trois nombres se répondent, et c'est tout l'argument du carnet :

- le seuil de dominance du risque, calculé **indépendamment** sur les produits de Nash : `0.7000000000` ;
- la limite de la coupure d'équilibre, obtenue **par résolution numérique** du jeu à signaux : `0.6999319` à `σ_ε = 0.001`, soit `6.8e-05` du précédent ;
- le seuil de dominance Pareto, `0.4000`, qui n'est **pas** la limite.

Deux procédures sans rapport — une comparaison de produits de pertes sur une matrice 2×2, et une bissection sur une équation de croyance — tombent sur le même point. La coïncidence n'est pas un hasard : c'est le théorème de Carlsson et van Damme, et ce carnet en vérifie l'arithmétique sur un jeu concret.

Sur la table, la bascule se lit entre `0.60` et `0.80`, avec l'égalité exacte en `0.70` : en dessous, `(N, N)` est risque-dominant alors que `(I, I)` rapporte davantage aux deux joueurs ; au-dessus, les deux critères s'accordent.

In [7]:

var inv = CultureInfo.InvariantCulture;
// figure retraced : c* en fonction du bruit, face aux deux seuils de dominance
Console.WriteLine("retrace de la figure : c*(sigma_eps) face aux seuils de dominance");
Console.WriteLine(" sigma_eps         c*");
foreach (double se in new[] { 1.0, 0.5, 0.3, 0.1, 0.05, 0.02, 0.01, 0.001 })
{
    double cs = Gt11f.Coupure(se);
    Console.WriteLine($"{se.ToString("F3", inv),10}   {cs.ToString("F6", inv),9}");
}
Console.WriteLine($"dominance du risque  c - gamma/2 = {seuilRisque.ToString("F2", inv)}");
Console.WriteLine($"dominance Pareto  c - gamma    = {(Gt11f.C - Gt11f.GAMMA).ToString("F2", inv)}");
Gt11f.AssertClose(Gt11f.Coupure(0.01), 0.6992780, 1e-5, "c*@0.01 (retrace)");
Gt11f.AssertClose(Gt11f.Coupure(1.0), -0.8632668, 1e-4, "c*@1.0 (retrace)");
Console.WriteLine("OK - retrace cellule 17 (figure -> table)");


retrace de la figure : c*(sigma_eps) face aux seuils de dominance


 sigma_eps         c*


     1.000   -0.863267


     0.500    0.474098


     0.300    0.626185


     0.100    0.688370


... (3 outputs left out)

     0.001    0.699932


dominance du risque  c - gamma/2 = 0.70


dominance Pareto  c - gamma    = 0.40


OK - retrace cellule 17 (figure -> table)


### Lecture du résultat

La table ci-dessus retrace la figure du jumeau Python. À bruit fort la coupure s'effondre très en dessous de la bande — un joueur qui ne sait presque rien n'investit que si le signal est spectaculairement bon. Quand le bruit s'éteint, la coupure se stabilise exactement sur le seuil de dominance du risque `c - γ/2`, à l'intérieur de la bande de multiplicité, **sans jamais rejoindre** le seuil de dominance Pareto `c - γ`.

C'est la réponse à la question du §1. La bande de multiplicité ne disparaît pas quand l'information se dégrade : elle est **remplacée** par un point unique, et ce point est le milieu de la bande. La sélection tient donc à la structure de l'information, non à un raffinement des payoffs — les payoffs, eux, n'ont pas changé d'un iota entre le §1 et ici.


### 4 bis — l'écart de 6,8e-05 : un artefact de résolution, ou un terme de premier ordre ?

La section 4 a mis en regard deux nombres : le seuil de dominance du risque
`c - γ/2 = 0.7000000000`, obtenu **indépendamment** sur les produits de Nash,
et la coupure d'équilibre du jeu à signaux `c* = 0.6999319` à `σ_ε = 0.001`,
soit `6.8e-05` d'écart. Les deux nombres « se répondent » : à bruit nul, la
coupure converge vers `c - γ/2`.

Reste à dire **ce qu'est** cet écart. Deux hypothèses, et une mesure qui les
départage :

- **H1 — un artefact de résolution.** Si l'écart venait de la bissection
  (`tol = 1e-13`), il devrait disparaître quand on raffine la résolution.
- **H2 — une propriété du modèle.** L'écart vient de la probabilité
  `p = P(l'autre investit | x_i = c*)`, qui, à signal imparfait, s'écarte de
  `1/2`. L'écart doit alors **survivre** à toute raffinement numérique, et
  **varier comme le bruit** : à `σ_ε → 0`, il doit s'annuler.

La cellule qui suit fait les deux choses : elle raffine la résolution (H1),
elle mesure l'écart en fonction du bruit (H2), et elle confronte le rapport
`écart / σ_ε` **mesuré** au coefficient **prédit** par le développement de
Taylor — une prédiction chiffrée, pas une concordance de façade.


In [8]:

var inv = CultureInfo.InvariantCulture;
Console.WriteLine("H1 -- l'ecart survit-il au raffinement de la resolution ?");
foreach (double tol in new[] { 1e-6, 1e-9, 1e-13, 1e-16 })
{
    double cs = Gt11f.Coupure(0.001, tol: tol);
    Console.WriteLine($"  tol = {tol.ToString("E0", inv)} : c* = {cs.ToString("F10", inv)}   ecart = {(cs - seuilRisque).ToString("E3", inv)}");
}
Console.WriteLine("  (tol = 1e-13 est la resolution du carnet ; les autres ne doivent rien changer)");
Console.WriteLine();
Console.WriteLine("H2 -- l'ecart en fonction du bruit : un terme de premier ordre ?");
Console.WriteLine("   sigma_eps            c*          ecart    ecart/sigma_eps");
foreach (double se in new[] { 0.1, 0.01, 0.001, 0.0001, 0.00001 })
{
    double cs = Gt11f.Coupure(se);
    double ec = cs - seuilRisque;
    Console.WriteLine($"{se.ToString("F5", inv),12} {cs.ToString("F10", inv),14} {ec.ToString("E6", inv),14} {(ec / se).ToString("F6", inv),17}");
}
Console.WriteLine();
double c0 = Gt11f.C - Gt11f.GAMMA / 2.0;
double w1 = (Gt11f.MU - c0) / (Gt11f.SIGMA_T * Gt11f.SIGMA_T * Math.Sqrt(2.0));
double coefficientPredit = -Gt11f.GAMMA * w1 / Math.Sqrt(2.0 * Math.PI);
double mesure = (Gt11f.Coupure(0.00001) - seuilRisque) / 0.00001;
Console.WriteLine($"coefficient predit  (developpement de Taylor) : {coefficientPredit.ToString("F6", inv)}");
Console.WriteLine($"coefficient mesure (ecart/sigma_eps a 1e-05)  : {mesure.ToString("F6", inv)}");
Console.WriteLine($"ecart relatif des deux                         : {(Math.Abs(mesure - coefficientPredit) / Math.Abs(coefficientPredit)).ToString("E2", inv)}");
Gt11f.AssertClose(coefficientPredit, -0.067703, 1e-5, "coefficient predit");
Gt11f.AssertClose(mesure, -0.067707, 1e-5, "coefficient mesure");
double ecartRel = Math.Abs(mesure - coefficientPredit) / Math.Abs(coefficientPredit);
Gt11f.AssertClose(ecartRel, 6.49e-05, 2e-5, "ecart relatif");
Console.WriteLine("OK - assertions cellule 20");


H1 -- l'ecart survit-il au raffinement de la resolution ?


  tol = 1E-006 : c* = 0.6999325752   ecart = -6.742E-005


  tol = 1E-009 : c* = 0.6999318511   ecart = -6.815E-005


  tol = 1E-013 : c* = 0.6999318510   ecart = -6.815E-005


  tol = 1E-016 : c* = 0.6999318510   ecart = -6.815E-005


  (tol = 1e-13 est la resolution du carnet ; les autres ne doivent rien changer)


... (7 outputs left out)

coefficient predit  (developpement de Taylor) : -0.067703


coefficient mesure (ecart/sigma_eps a 1e-05)  : -0.067707


ecart relatif des deux                         : 6.49E-005


OK - assertions cellule 20


### Lecture du résultat

Les deux hypothèses sont départagées, et c'est **H2** qui l'emporte.

**L'écart n'est pas un artefact de résolution.** Raffiner la bissection ne
le fait pas disparaître : à `tol = 1e-9`, `1e-13` et `1e-16`, la coupure
vaut `0.699931851` à **neuf** décimales — à la dixième, seules `tol = 1e-13`
et `tol = 1e-16` coïncident —, et l'écart `-6.815e-05` reste **le même
chiffre**. Mais il faut dire à quelle précision : ces trois valeurs ne sont
**pas** identiques au 1e-13 près. Lues à pleine précision, `tol = 1e-9` donne
`0.6999318511225283` et s'écarte de **7,8e-11** des deux autres, qui ne
diffèrent entre elles que de **5,4e-14** (`tol = 1e-13` contre `tol = 1e-16`).
La bissection est donc convergente **au 1e-13 près** dès `tol = 1e-13`, et
l'écart mesuré est stable **au 1e-10 près** entre les trois tolérances — six
ordres de grandeur sous l'écart lui-même. Seul `tol = 1e-6` s'en écarte
(`-6.742e-05`) : c'est là, et seulement là, que la résolution numérique
devient visible — `tol = 1e-6` est mille fois plus grossière que la
tolérance de référence (`1e-9`), et son effet sur la coupure (`7,2e-07`)
reste environ cent fois **plus petit** que l'écart mesuré (`6,8e-05`). Un
artefact de résolution aurait disparu ; il est resté.

> **Comment lire ce qui précède.** La cellule affiche `c*` à dix décimales
> (`0.6999318511` pour `tol = 1e-9`, `0.6999318510` pour `tol = 1e-13` et
> `tol = 1e-16`) : l'accord des trois valeurs s'arrête à la neuvième ;
> l'écart de `7,8e-11` ne se lit pas sur cette sortie — il se mesure à pleine
> précision : `coupure(0.001, tol=1e-9) - coupure(0.001, tol=1e-13)` vaut
> `-7,8e-11`. C'est la commande à rejouer pour vérifier la phrase ci-dessus.

**L'écart est un terme de premier ordre en bruit.** Le rapport
`écart / σ_ε` se stabilise : `-0.116302` à `σ_ε = 0.1`, `-0.072198` à
`0.01`, `-0.068149` à `0.001`, `-0.067747` à `1e-4`, `-0.067707` à `1e-5`.
Il tend vers une **constante**, non vers zéro : l'écart vaut
`≈ -0.0677 · σ_ε` — il s'annule avec le bruit, mais **proportionnellement**.

**Et la constante était prédite.** Le développement de Taylor de
`p = Φ(w)` autour de la limite `σ_ε → 0` donne, **au premier ordre**, le
coefficient `-γ · w₁ / √(2π) = -0.067703`, avec `w₁ = (μ - c₀)/(σ_θ²√2)`.
Le rapport **mesuré** à `σ_ε = 1e-5` vaut `-0.067707` : l'écart relatif des
deux est `6.49e-05`. La mesure ne « colle » pas au modèle après coup — elle
**vérifie une prédiction chiffrée**.

> **Ce que l'égalité autorise, et jusqu'où.** L'égalité est une égalité **au
> premier ordre** : `écart/σ_ε → -γ · w₁ / √(2π)` quand `σ_ε → 0`, avec un
> reste d'ordre `σ_ε` — les termes d'ordre `σ_ε²` du développement de `w`
> en font un `O(σ_ε)` sur le rapport. Le désaccord relatif mesuré, `6.49e-05`
> à `σ_ε = 1e-5`, est bien de cet ordre (≈ `6,5 · σ_ε`) : il **décroît avec
> le bruit** au lieu de s'annuler comme le ferait une erreur de modèle —
> c'est la signature d'un reste de développement d'ordre `σ_ε`. Lire
> `écart = -0.0677 · σ_ε` comme une **égalité** serait excessif : la forme
> exacte est `écart = -0.0677 · σ_ε + O(σ_ε²)`.

**Ce que cela dit du carnet.** La section 4 posait que les deux nombres se
répondent : à bruit nul, la coupure d'équilibre converge vers
`c - γ/2 = 0.7000000000`. C'est confirmé. Et le « `6.8e-05` » n'est pas un
bruit à ignorer : c'est la **correction de premier ordre** qui sépare le jeu
à signaux du seuil de dominance du risque. Le signe est net — à bruit fini
la coupure est **en dessous** du seuil (`0.6999319 < 0.7000000`) : le jeu
bruité sélectionne plus fortement le risque-dominant que sa limite. Lire
« seuil de dominance du risque = coupure d'équilibre » serait donc faux dès
que le bruit est positif ; la bonne formulation est « le seuil de dominance
du risque est la **limite** de la coupure d'équilibre, et l'écart au bruit
`σ_ε` est le terme de premier ordre `-0.0677 · σ_ε`, plus un `O(σ_ε²)` ».


## 5. La sélection, mesurée sous coordination perturbée

Les sections 3 et 4 ont établi une sélection **statique** : la coupure `c*` se déplace vers `c - γ/2` à mesure que le bruit privé s'éteint, et à la limite c'est l'équilibre Pareto-dominé qui survit. Ce résultat porte sur **l'équilibre** — sur ce qui existe — jamais sur le **chemin** qui y mène.

Savoir quel équilibre existe et savoir si l'on y arrive sont deux questions distinctes. Une sélection *affirmée* n'est pas une sélection *mesurée* : il manque l'observation d'un ensemble de trajectoires dont on regarde où elles se terminent.

### Ce qui perturbe la coordination

Dans un jeu de complémentarité, l'histoire d'un équilibre est faite de croyances partagées. La perturbation la plus directe n'est donc pas un choc sur les gains, mais **l'état initial de ces croyances** : ici, le taux `π₀` de la population qui investit au premier tour.

Trois issues sont possibles, et elles ne se distinguent qu'à la mesure :

| Ce qu'on observe | Ce que cela veut dire |
|---|---|
| toutes les valeurs de `π₀` mènent au même équilibre | la sélection est **nette** — le bruit privé a effacé l'histoire |
| deux valeurs de `π₀` mènent à deux équilibres purs différents | la sélection est **partielle** — deux bassins coexistent, l'histoire décide |
| aucune trajectoire ne se fixe sur un équilibre pur | il n'y a **aucune sélection** à observer |

### La dynamique retenue

Chaque joueur observe le taux agrégé `π` du tour précédent, forme `E[θ | x_i]`, et investit si l'écart anticipé reste positif compte tenu de `π` :

$$\text{investit} \iff \hat\theta_i - c + \gamma\,\pi \ge 0$$

Le seuil de signal correspondant est `x*(π) = μ + (c - γπ - μ)/λ`, d'où le taux induit au tour suivant :

$$F(\pi) = \Phi\!\left(\frac{\theta - x^*(\pi)}{\sigma_\varepsilon}\right)$$

Les équilibres de cette dynamique sont les **points fixes de `F`**. `F` est croissante : quand plus de gens investissent, chacun investit plus volontiers. Une application croissante d'un intervalle dans lui-même n'oscille pas — elle converge, et **lequel des points fixes elle atteint dépend du point de départ dès qu'il y en a plusieurs**. C'est exactement ce que la perturbation de coordination met à l'épreuve.

> **Ce que cette dynamique est, et ce qu'elle n'est pas.** C'est une meilleure réponse sur le taux *agrégé* — un raisonnement de niveau `k`, pas l'équilibre bayésien de Nash du §3. Ses points fixes ne coïncident avec la coupure du §3 que dans le cas où la fraction investissant à l'équilibre vaut exactement `1/2`. La cellule qui suit mesure cet écart plutôt que de le supposer nul.

In [9]:

var inv = CultureInfo.InvariantCulture;
double[] PI0S = { 0.05, 0.25, 0.50, 0.75, 0.95 };
double[] BRUITS = { 0.60, 0.10, 0.01 };
Console.WriteLine("  theta  sigma_eps    c* (S3)   F(0)   F(1)   taux terminal pour pi0 = 0.05, 0.25, 0.50, 0.75, 0.95");
Console.WriteLine(new string('-', 106));
foreach (double theta in new[] { 0.55, 0.85 })
{
    foreach (double se in BRUITS)
    {
        double[] fins = new double[PI0S.Length];
        for (int i = 0; i < PI0S.Length; i++)
        {
            var suite = Gt11f.Trajectoire(theta, se, PI0S[i]);
            fins[i] = suite[suite.Count - 1];
        }
        string finsStr = string.Join("  ", fins.Select(f => f.ToString("F4", inv).PadLeft(7)));
        Console.WriteLine($"{theta.ToString("F2", inv),6} {se.ToString("F2", inv),10} {Gt11f.Coupure(se).ToString("F6", inv),10} " +
            $"{Gt11f.TauxInduit(0.0, theta, se).ToString("F4", inv),8} {Gt11f.TauxInduit(1.0, theta, se).ToString("F4", inv),8}   {finsStr}");
    }
    Console.WriteLine();
}
var tr1 = Gt11f.Trajectoire(0.55, 0.60, 0.05);
var tr2 = Gt11f.Trajectoire(0.85, 0.60, 0.05);
var tr3 = Gt11f.Trajectoire(0.55, 0.01, 0.05);
Gt11f.AssertClose(tr1[tr1.Count - 1], 0.3957, 1e-3, "taux terminal th=0.55 se=0.60");
Gt11f.AssertClose(tr2[tr2.Count - 1], 0.9415, 1e-3, "taux terminal th=0.85 se=0.60");
Gt11f.AssertClose(tr3[tr3.Count - 1], 0.0, 1e-3, "taux terminal th=0.55 se=0.01 pi0=0.05");
Console.WriteLine("OK - assertions cellule 23");


  theta  sigma_eps    c* (S3)   F(0)   F(1)   taux terminal pour pi0 = 0.05, 0.25, 0.50, 0.75, 0.95


----------------------------------------------------------------------------------------------------------


  0.55       0.60   0.337658   0.1093   0.8869    0.3957   0.3957   0.3957   0.3957   0.3957


  0.55       0.10   0.688370   0.0000   0.9515    0.0000   0.0000   0.0000   0.0000   0.0000


  0.55       0.01   0.699278   0.0000   1.0000    0.0000   0.0000   0.0000   0.0000   1.0000


  0.85       0.60   0.337658   0.2327   0.9564    0.9415   0.9415   0.9415   0.9415   0.9415


  0.85       0.10   0.688370   0.0571   1.0000    1.0000   1.0000   1.0000   1.0000   1.0000


  0.85       0.01   0.699278   0.0000   1.0000    0.0000   1.0000   1.0000   1.0000   1.0000


OK - assertions cellule 23


### Lecture du résultat

Deux colonnes expliquent tout le reste : `F(0)` et `F(1)`. Un équilibre pur survit si et seulement si `F` **le fixe** — `F(0) = 0` pour `(N,N)`, `F(1) = 1` pour `(I,I)`.

| `σ_ε` | `θ = 0.55` | `θ = 0.85` | ce que la dynamique peut faire |
|---|---|---|---|
| `0.60` | `F(0) = 0.109`, `F(1) = 0.887` | `F(0) = 0.233`, `F(1) = 0.956` | **aucun** équilibre pur n'est un point fixe |
| `0.10` | `F(0) = 0.000`, `F(1) = 0.952` | `F(0) = 0.057`, `F(1) = 1.000` | un seul équilibre pur survit |
| `0.01` | `F(0) = 0.000`, `F(1) = 1.000` | `F(0) = 0.000`, `F(1) = 1.000` | **deux** points fixes purs coexistent |

Trois régimes, et ils ne se lisent pas dans `θ` seul :

- **Bruit fort (`0.60`)** — `F` reste loin de ses deux extrémités. Les cinq valeurs de `π₀` convergent vers le **même** point fixe, mais ce point est **intérieur** : `0.3957` à `θ = 0.55`, `0.9415` à `θ = 0.85`. Une fraction seulement de la population investit. Ce n'est pas une sélection entre deux équilibres purs — c'est l'absence d'équilibre pur.
- **Bruit intermédiaire (`0.10`)** — l'extrémité opposée au risque-dominant a cessé d'être un point fixe (`F(0) = 0.057 > 0` à `θ = 0.85` ; `F(1) = 0.952 < 1` à `θ = 0.55`), et toutes les trajectoires convergent vers l'équilibre risque-dominant : `(N,N)` à `θ = 0.55`, sous le seuil `c - γ/2 = 0.7` ; `(I,I)` à `θ = 0.85`, au-dessus. Une sélection **nette**, du côté prédit par la dominance du risque.
- **Bruit faible (`0.01`)** — `F(0) = 0` **et** `F(1) = 1` : les deux équilibres purs sont des points fixes, séparés par un troisième point fixe instable. C'est la **bistabilité**. Une seule valeur de `π₀` échappe au risque-dominant — `0.95` à `θ = 0.55`, `0.05` à `θ = 0.85` — et finit de l'autre côté.

Le résultat est un renversement, et c'est le point de la section : **c'est quand le bruit privé s'éteint que le passé redevient déterminant.** À bruit fort, tout converge au même endroit, mais cet endroit n'est plus un équilibre pur — l'unicité ne sélectionne donc rien entre deux équilibres. À bruit faible, deux équilibres purs cohabitent et `π₀` tranche. La sélection est nette **entre** les deux, à bruit intermédiaire — là précisément où `F` ne coupe la diagonale qu'une fois.

In [10]:

var inv = CultureInfo.InvariantCulture;
Console.WriteLine(" sigma_eps    c* (S3)    pi_3   F(pi_3)  F(pi_3) - pi_3");
Console.WriteLine(new string('-', 56));
foreach (double se in new[] { 0.80, 0.50, 0.30, 0.20, 0.10, 0.05, 0.02, 0.01 })
{
    double cs = Gt11f.Coupure(se);
    double pi3 = Gt11f.Phi((Gt11f.MU - cs) / Math.Sqrt(Gt11f.SIGMA_T * Gt11f.SIGMA_T + se * se));
    double fpi3 = Gt11f.TauxInduit(pi3, Gt11f.MU, se);
    Console.WriteLine($"{se.ToString("F3", inv),10} {cs.ToString("F6", inv),10} {pi3.ToString("F4", inv),8} {fpi3.ToString("F4", inv),9} {(fpi3 - pi3).ToString("F4", inv),15}");
}
double cs80 = Gt11f.Coupure(0.80);
double pi380 = Gt11f.Phi((Gt11f.MU - cs80) / Math.Sqrt(Gt11f.SIGMA_T * Gt11f.SIGMA_T + 0.64));
Gt11f.AssertClose(pi380, 0.8393, 1e-3, "pi_3 @ se=0.80");
Gt11f.AssertClose(Gt11f.TauxInduit(pi380, Gt11f.MU, 0.80), 0.9116, 1e-3, "F(pi_3) @ se=0.80");
Console.WriteLine("OK - assertions cellule 25");


 sigma_eps    c* (S3)    pi_3   F(pi_3)  F(pi_3) - pi_3


--------------------------------------------------------


     0.800  -0.135308   0.8393    0.9116          0.0724


     0.500   0.474098   0.6776    0.7956          0.1181


     0.300   0.626185   0.6172    0.7800          0.1628


     0.200   0.665081   0.5989    0.8223          0.2234


     0.100   0.688370   0.5866    0.9430          0.3564


     0.050   0.695452   0.5824    0.9987          0.4163


     0.020   0.698465   0.5804    1.0000          0.4196


     0.010   0.699278   0.5798    1.0000          0.4202


OK - assertions cellule 25


### Lecture du résultat

La dernière colonne répond à la question posée : **non, ce ne sont pas les mêmes équilibres.**

- `π₃` reste entre `0.58` et `0.84` quand le bruit décroît de `0.80` à `0.01` — il ne s'approche jamais de `1/2` ;
- `F(π₃) - π₃` est **positif partout**, et grandit jusqu'à `0.42` quand le bruit s'éteint.

Autrement dit, à la coupure du §3, la dynamique agrégée **pousse encore vers le haut** : ce point n'est pas un point fixe, c'est un point de passage. Les deux objets décrivent le même jeu, mais pas le même équilibre — la coupure du §3 est l'équilibre bayésien de Nash symétrique, la meilleure réponse agrégée converge vers un autre profil, où davantage de gens investissent.

L'écart ne se referme pas quand `σ_ε → 0`, et c'est le point le plus utile de cette cellule : la limite `c - γ/2` est un résultat sur la **coupure**, et elle ne se transporte pas telle quelle sur la **fraction agrégée**. Les deux ne coïncideraient que si `π₃ = 1/2` — si chaque joueur croyait qu'exactement un autre sur deux investit. Ce n'est vrai dans aucun des cas mesurés.

> **Conséquence de lecture.** La suite mesure la sélection sur la dynamique agrégée, celle qu'on peut simuler et observer. Ses verdicts portent donc sur *cette* dynamique et ne valent pas comme confirmation numérique de la coupure du §3. Le carnet gagne ici une distinction qu'il n'avait pas : l'unicité de l'équilibre bayésien et la convergence d'une dynamique de coordination sont deux propriétés indépendantes.

In [11]:

var inv = CultureInfo.InvariantCulture;
// Monte-Carlo : les tirages (graines 0,1,7,42, N=200) viennent du jumeau Python et sont CITES
Console.WriteLine("Monte-Carlo [CITE du jumeau Python - tirages non rejoues ici]");
Console.WriteLine("  theta  sigma_eps |  (I,I)  (N,N)  interieur | pi min pi max | verdict");
Console.WriteLine(new string('-', 96));
var cite = new (double th, double se, int nIi, int nNn, int nInt, double pmin, double pmax, string verdict)[]
{
    (0.55, 0.60, 0, 0, 20, 0.080, 0.725, "aucune selection"),
    (0.55, 0.10, 0, 20, 0, 0.000, 1.000, "selection nette"),
    (0.55, 0.01, 4, 16, 0, 0.000, 1.000, "selection partielle"),
    (0.85, 0.60, 0, 0, 20, 0.945, 0.970, "aucune selection"),
    (0.85, 0.10, 19, 0, 1, 0.055, 1.000, "selection partielle"),
    (0.85, 0.01, 16, 4, 0, 0.000, 1.000, "selection partielle"),
};
foreach (var r in cite)
{
    Console.WriteLine($"{r.th.ToString("F2", inv),6} {r.se.ToString("F2", inv),10} | {r.nIi,6} {r.nNn,6} {r.nInt,10} | {r.pmin.ToString("F3", inv),7} {r.pmax.ToString("F3", inv),7} | {r.verdict}");
}
// verification interne : les comptes cites sont coherents avec le verdict
foreach (var r in cite)
{
    int n = r.nIi + r.nNn + r.nInt;
    if (n != 20) throw new Exception("MC cite : compte incoherent (!= 20)");
    string attendu = (r.nIi == 20 || r.nNn == 20) ? "selection nette" : (r.nInt == 20 ? "aucune selection" : "selection partielle");
    if (attendu != r.verdict) throw new Exception($"MC cite : verdict incoherent ({r.verdict} vs {attendu})");
}
Console.WriteLine("OK - verification interne des comptes cites (cellule 27)");


Monte-Carlo [CITE du jumeau Python - tirages non rejoues ici]


  theta  sigma_eps |  (I,I)  (N,N)  interieur | pi min pi max | verdict


------------------------------------------------------------------------------------------------


  0.55       0.60 |      0      0         20 |   0.080   0.725 | aucune selection


  0.55       0.10 |      0     20          0 |   0.000   1.000 | selection nette


  0.55       0.01 |      4     16          0 |   0.000   1.000 | selection partielle


  0.85       0.60 |      0      0         20 |   0.945   0.970 | aucune selection


  0.85       0.10 |     19      0          1 |   0.055   1.000 | selection partielle


  0.85       0.01 |     16      4          0 |   0.000   1.000 | selection partielle


OK - verification interne des comptes cites (cellule 27)


### Lecture du résultat

Le Monte-Carlo retrouve les trois régimes, et ajoute ce que le champ moyen ne peut pas dire : la **dispersion**. Les tirages (graines `0, 1, 7, 42`, `N = 200`) sont **CITÉS du jumeau Python** — non rejoués ici ; la cellule ci-dessus vérifie leur cohérence interne (comptes et verdicts).

| `θ` | `σ_ε` | issue dominante | verdict |
|---|---|---|---|
| `0.55` | `0.60` | 20/20 intérieur, `π` de `0.080` à `0.725` | aucune sélection |
| `0.55` | `0.10` | 20/20 `(N,N)` | sélection nette |
| `0.55` | `0.01` | 16/20 `(N,N)`, 4/20 `(I,I)` | sélection partielle |
| `0.85` | `0.60` | 20/20 intérieur, `π` de `0.945` à `0.970` | aucune sélection |
| `0.85` | `0.10` | 19/20 `(I,I)`, 1/20 intérieur (`0.055`) | sélection partielle |
| `0.85` | `0.01` | 16/20 `(I,I)`, 4/20 `(N,N)` | sélection partielle |

Trois choses que seule la population finie montre :

1. **Le bassin du risque-dominant est le plus large — pas total.** À `σ_ε = 0.01`, le rapport est de **16 contre 4** dans les deux sens : l'équilibre prédit par la dominance du risque l'emporte quatre fois sur cinq. Une sélection forte, pas une sélection absolue. Le champ moyen annonçait le même 4/5 par le compte des points de départ (un seul `π₀` sur cinq bascule) — deux mesures indépendantes, même verdict.
2. **La dispersion à bruit fort est énorme.** À `θ = 0.55`, `σ_ε = 0.60`, le taux terminal va de `0.080` à `0.725` selon la graine et le point de départ. Le champ moyen donnait **un** point fixe à `0.3957` ; la population finie en admet une plage, parce que la meilleure réponse y est un **escalier** : ses marches plates coupent la diagonale plusieurs fois, et `π₀` choisit la marche.
3. **La population finie peut se verrouiller sur le mauvais équilibre.** Le `1/20` de la ligne `θ = 0.85, `σ_ε` = 0.10` s'arrête à `0.055` là où le champ moyen envoie tout le monde à `1.0` : une graine dont l'échantillon place onze joueurs du mauvais côté produit un seuil qui se confirme lui-même. Avec `N = 200`, la granularité de `1/200` suffit à figer une minorité auto-réalisatrice.

La réponse honnête à la question de départ tient en trois lignes : **la sélection existe, elle est orientée par la dominance du risque, et elle n'est jamais certaine.** Le bruit privé décide du *régime* ; le point de départ et l'échantillon décident du sort *à l'intérieur* du régime. Aucune des deux mesures ne permet d'annoncer un équilibre terminal sans dire d'où l'on part.


## 6. Le raccourci qui ne marche pas

L'équation d'indifférence du §3 s'écrit souvent, dans les présentations rapides, sous une forme close :

$$
c^* = c - \gamma \, p, \qquad p = \mathbb{P}(x_j \geq c^* \mid x_i = c^*)
$$

C'est une **erreur**, et une erreur instructive. Cette relation suppose `E[θ | c*] = c*`, c'est-à-dire `λ = 1` : elle traite le joueur au seuil comme s'il **connaissait** `θ`. Or `λ < 1` tant que le bruit privé existe, et `E[θ | c*] = μ + λ(c* - μ)` s'écarte de `c*` d'autant plus que le signal est peu informatif. La forme close ne vaut donc qu'à la limite `σ_ε = 0` — précisément le cas où il n'y a plus de jeu global.

La cellule suivante mesure l'écart entre les deux relations.

In [12]:

var inv = CultureInfo.InvariantCulture;
Console.WriteLine(" sigma_eps    c* exact        p   c - gamma*p        ecart");
foreach (double se in new[] { 1.0, 0.5, 0.3, 0.2, 0.1, 0.05, 0.01, 0.001 })
{
    double cs = Gt11f.Coupure(se);
    double p = Gt11f.ProbaAutre(cs, se);
    double naif = Gt11f.C - Gt11f.GAMMA * p;
    double ec = naif - cs;
    string ecStr = (ec >= 0 ? "+" : "") + ec.ToString("E2", inv);
    Console.WriteLine($"{se.ToString("F3", inv),10} {cs.ToString("F7", inv),12} {p.ToString("F6", inv),10} {naif.ToString("F7", inv),13} {ecStr,11}");
}
Gt11f.AssertClose(Gt11f.Coupure(1.0), -0.8632668, 1e-4, "c*@1.0");
double naif001 = Gt11f.C - Gt11f.GAMMA * Gt11f.ProbaAutre(Gt11f.Coupure(0.001), 0.001);
Gt11f.AssertClose(naif001 - Gt11f.Coupure(0.001), 4.00e-07, 1e-6, "ecart @0.001");
Console.WriteLine("OK - assertions cellule 30");


 sigma_eps    c* exact        p   c - gamma*p        ecart


     1.000   -0.8632668   0.887756     0.4673466  +1.33E+000


     0.500    0.4740985   0.604918     0.6370492  +1.63E-001


     0.300    0.6261851   0.546342     0.6721949  +4.60E-002


     0.200    0.6650814   0.527182     0.6836909  +1.86E-002


     0.100    0.6883698   0.512228     0.6926633  +4.29E-003


     0.050    0.6954522   0.505854     0.6964874  +1.04E-003


     0.010    0.6992780   0.501136     0.6993183  +4.03E-005


     0.001    0.6999319   0.500113     0.6999323  +4.00E-007


OK - assertions cellule 30


### Lecture du résultat

Le raccourci est faux **là où le jeu global a un intérêt**, et il devient exact là où il n'en a plus :

- à `σ_ε = 1.0`, il se trompe de `+1.33` — plus que la largeur entière de la bande ;
- à `σ_ε = 0.5`, encore de `+0.163`, soit un quart de cette largeur ;
- à `σ_ε = 0.001`, l'écart est de `4.0e-07` : le bruit est devenu négligeable, `λ ≈ 1`, et la forme close rejoint la vraie.

Autrement dit, l'erreur n'est pas une faute de calcul mais une **hypothèse implicite** : la forme close suppose résolue la question que le carnet pose. C'est pourquoi le §3 garde l'espérance conditionnelle `E[θ | c*]` dans l'équation plutôt que de la remplacer par `c*`.

## 7. Exercices

Les trois exercices reprennent, dans l'ordre, les trois mécanismes du carnet : l'équation d'indifférence, le critère de sélection, et l'unicité. Les cellules ci-dessous sont des squelettes — elles s'exécutent sans erreur et renvoient `None` tant qu'elles ne sont pas complétées.

**Exercice 1 — résoudre l'équation d'indifférence.** Implémenter `coupure_equilibre` par bissection, en réutilisant `ecart_indifference`, et vérifier que la valeur obtenue pour `sigma_eps = 0.05` reproduit celle du tableau du §3.

In [13]:

public static class Exo
{
    // Exercice 1 — coupure d'equilibre, par bissection.
    // TODO etudiant
    public static double? CoupureEquilibre(double sigmaEps, double lo = -20.0, double hi = 20.0, double tol = 1e-13)
    {
        return null;
    }

    // Exercice 2 — intervalle ou dominance Pareto et dominance du risque divergent.
    // TODO etudiant
    public static (double BorneBasse, double BorneHaute)? Conflit(double c, double gamma)
    {
        return null;
    }

    // Exercice 3 — iterer la meilleure reponse jusqu'au point fixe.
    // TODO etudiant
    public static double? PointFixeParIteration(double depart, double sigmaEps, int maxIter = 500, double tol = 1e-14)
    {
        return null;
    }
}

var valeur = Exo.CoupureEquilibre(0.05);
Console.WriteLine("coupure obtenue pour sigma_eps = 0.05 : " + (valeur?.ToString() ?? "null"));
Console.WriteLine("a comparer a la ligne 0.050 du tableau de la section 3.");


coupure obtenue pour sigma_eps = 0.05 : null


a comparer a la ligne 0.050 du tableau de la section 3.


**Exercice 2 — confronter les deux critères.** Écrire `conflit(c, gamma)` qui renvoie l'intervalle `[c - gamma, c - gamma/2]` sur lequel la dominance Pareto et la dominance du risque désignent des équilibres **différents**. Vérifier que sa largeur vaut `gamma/2`, puis expliquer en commentaire lequel des deux critères le jeu global suit, et pourquoi la réponse ne dépend pas de `sigma_theta`.

In [14]:

var intervalle = Exo.Conflit(Gt11f.C, Gt11f.GAMMA);
Console.WriteLine("intervalle de conflit : " + (intervalle?.ToString() ?? "null"));
Console.WriteLine("largeur attendue gamma/2 = " + (Gt11f.GAMMA / 2).ToString(CultureInfo.InvariantCulture));


intervalle de conflit : null


largeur attendue gamma/2 = 0.3


**Exercice 3 — établir l'unicité par itération.** Écrire une fonction qui, partant d'un seuil arbitraire, itère `meilleure_reponse` jusqu'au point fixe. L'appliquer à quatre départs éloignés pour deux niveaux de bruit, et vérifier que l'écart entre les points d'arrivée est de l'ordre de `1e-13`. Conclure : combien de points fixes cette procédure trouve-t-elle, et qu'est-ce que cela dit du nombre d'équilibres ?

In [15]:

foreach (double bruit in new[] { 0.3, 0.02 })
{
    double[] departs = { -3.0, 0.0, 1.0, 2.5 };
    var arrives = new List<string>();
    foreach (double d in departs)
        arrives.Add(Exo.PointFixeParIteration(d, bruit)?.ToString() ?? "null");
    Console.WriteLine($"sigma_eps = {bruit.ToString(CultureInfo.InvariantCulture)} : departs [-3, 0, 1, 2.5] -> arrives [{string.Join(", ", arrives)}]");
}


sigma_eps = 0.3 : departs [-3, 0, 1, 2.5] -> arrives [null, null, null, null]


sigma_eps = 0.02 : departs [-3, 0, 1, 2.5] -> arrives [null, null, null, null]


## Références

- **Carlsson, H. & van Damme, E. (1993)** — « Global Games and Equilibrium Selection », *Econometrica*, 61(5), 989–1018. Le résultat fondateur : pour les jeux 2×2, quand le bruit s'éteint, l'élimination itérée des stratégies dominées dans le jeu global contraint les joueurs à suivre le critère de dominance du risque de Harsanyi et Selten.
- **Morris, S. & Shin, H. S. (1998)** — « Unique Equilibrium in a Model of Self-Fulfilling Currency Attacks », *American Economic Review*, 88(3), 587–597. La version à grand nombre de joueurs, où la même mécanique sélectionne l'issue d'une attaque spéculative.
- **Harsanyi, J. C. (1967–1968)** — « Games with Incomplete Information Played by "Bayesian" Players », Parties I–III, *Management Science*, 14. La transformation d'un jeu à information incomplète en jeu à information imparfaite dont l'équilibre de Nash est celui que ce carnet calcule (voir aussi le carnet 11).

## Portée de ce carnet

Ce que les cellules établissent directement : sur ce jeu précis, la condition d'indifférence est strictement croissante et son itération de meilleure réponse converge vers un point fixe unique ; la limite de la coupure quand le bruit s'éteint coïncide, à `7e-05` près, avec le seuil de dominance du risque calculé séparément sur les produits de Nash, et non avec le seuil de dominance Pareto.

Ce qu'elles n'établissent pas : le théorème lui-même. Carlsson et van Damme le démontrent par élimination itérée des stratégies dominées, pour toute la classe des jeux globaux — la mesure numérique d'un cas ne remplace pas cette preuve, elle en rend le mécanisme visible.